# Logo Recognition Clustring

This project aims to address the challenge of logo recognition and similarity search in the context of limited available datasets. Due to the vast diversity of logos and the constant emergence of new designs, traditional logo recognition systems often struggle to maintain high accuracy and robustness. To tackle this issue, the project proposes the use of synthetic data generation techniques alongside advanced deep learning models to significantly improve the system's ability to accurately identify and find similar logos.



## Imports

In [ ]:
# This get the RAPIDS-Colab install files and test check your GPU.  Run this and the next cell only.
# Please read the output of this cell.  If your Colab Instance is not RAPIDS compatible, it will warn you and give you remediation steps.
!git clone https://github.com/rapidsai/rapidsai-csp-utils.git
!python rapidsai-csp-utils/colab/pip-install.py


Cloning into 'rapidsai-csp-utils'...
remote: Enumerating objects: 587, done.
remote: Counting objects: 100% (153/153), done.
remote: Compressing objects: 100% (71/71), done.
remote: Total 587 (delta 122), reused 82 (delta 82), pack-reused 434 (from 3)
Receiving objects: 100% (587/587), 192.90 KiB | 2.57 MiB/s, done.
Resolving deltas: 100% (296/296), done.
Installing RAPIDS remaining 25.02 libraries
Using Python 3.11.11 environment at: /usr
Resolved 159 packages in 863ms
 Downloaded cuproj-cu12
 Downloaded cuspatial-cu12
 Downloaded datashader
 Downloaded cugraph-cu12
 Downloaded libcuspatial-cu12
 Downloaded cucim-cu12
Prepared 10 packages in 658ms
Installed 10 packages in 14ms
 + cucim-cu12==25.2.0
 + cugraph-cu12==25.2.0
 + cuproj-cu12==25.2.0
 + cuspatial-cu12==25.2.0
 + cuxfilter-cu12==25.2.0
 + datashader==0.17.0
 + jupyter-server-proxy==4.4.0
 + libcuspatial-cu12==25.2.0
 + pyct==0.5.0
 + simpervisor==1.0.0

        ****************************************************************

In [ ]:
import cudf
import cuml
import cupy as cp
from cuml.common.device_selection import using_device_type, set_global_device_type


(
    cudf.__version__,
    cuml.__version__,
    cp.__version__

 )

('25.02.01', '25.02.01', '13.3.0')

In [ ]:
!pip install datasets -q

import os
import numpy as np
import pandas as pd
from tqdm import tqdm
tqdm.pandas()
import torch
import datasets
import warnings
warnings.filterwarnings('ignore')


output_dir = f"/content/drive/MyDrive/logo_recognition_similarity_search_project/output"
os.makedirs(output_dir,exist_ok=True)
ds_dir = f"{output_dir}/logo-recognition-embeddings"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 487.4/487.4 kB 29.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 18.9 MB/s eta 0:00:00


## Load Dataset

In [ ]:
ds = datasets.load_from_disk(ds_dir)
ds = ds.with_format("numpy")
ds

Loading dataset from disk:   0%|          | 0/36 [00:00<?, ?it/s]

Dataset({
    features: ['id', 'path', 'prompt', 'prompt_embedding', 'image_embedding'],
    num_rows: 1777584
})

In [ ]:
ds.features

{'id': Value(dtype='string', id=None),
 'path': Value(dtype='string', id=None),
 'prompt': Value(dtype='string', id=None),
 'prompt_embedding': Sequence(feature=Value(dtype='float32', id=None), length=-1, id=None),
 'image_embedding': Sequence(feature=Value(dtype='float32', id=None), length=-1, id=None)}

In [ ]:
# Extract image and text embeddings
image_embeddings = np.array(ds['image_embedding'])
prompt_embeddings = np.array(ds['prompt_embedding'])

# Concatenate image and text embeddings
combined_embeddings = np.hstack((image_embeddings, prompt_embeddings))

del image_embeddings
del prompt_embeddings

combined_embeddings.shape

(1777584, 2432)

## Clustring

### UMAP

This section applies **UMAP** (Uniform Manifold Approximation and Projection) to the combined image and text embeddings. **UMAP** is a dimensionality reduction technique well-suited for visualizing high-dimensional data while preserving the underlying data structure.  The reduced embeddings are then used for clustering.  The code in this section assumes the existence of a preprocessed dataset with combined embeddings as generated in earlier parts of the notebooks.

In [ ]:
n_components=600

In [ ]:
from cuml.manifold import UMAP

umap_model = UMAP(
                  n_components=n_components,
                  metric='cosine',
                  min_dist=0.0,
                  random_state=42,
                  )

[2025-03-15 20:07:48.723] [CUML] [info] build_algo set to brute_force_knn because random_state is given


In [ ]:
reduced_embeddings = umap_model.fit_transform(combined_embeddings)

In [ ]:
reduced_embeddings.shape

(1777584, 600)

In [ ]:
np.save(f"{output_dir}/reduced_{n_components}_embeddings.npy", reduced_embeddings)

In [ ]:
reduced_embeddings = np.load(f"{output_dir}/reduced_{n_components}_embeddings.npy")
reduced_embeddings.shape

(1777584, 600)

### HDBSCAN

**HDBSCAN** (Hierarchical Density-Based Spatial Clustering of Applications with Noise) is a clustering algorithm that is well-suited for data with varying densities.  Unlike K-Means, which assumes spherical clusters of equal density, HDBSCAN can identify clusters of arbitrary shapes and sizes.  It's particularly effective at finding clusters in noisy data and identifying outliers.  We will use HDBSCAN to cluster the reduced embeddings.


In [ ]:
from cuml.cluster.hdbscan import HDBSCAN

# A higher min_cluster_size will generate fewer clusters.
# A lower min_cluster_size will generate more clusters.

hdbscan_model = HDBSCAN(min_cluster_size=2,
                        cluster_selection_epsilon=0.005,
                        metric='euclidean',
                        cluster_selection_method='eom',
                        prediction_data=True,

                        )

In [ ]:
hdbscan_model.fit(reduced_embeddings)

HDBSCAN()

In [ ]:
hdbscan_model.labels_.shape

(1777584,)

In [ ]:
hdbscan_model.n_clusters_

209286

In [ ]:
unique_labels, counts = np.unique(hdbscan_model.labels_, return_counts=True)
print(f"Number of unique labels: {len(unique_labels)}")
label_counts_df = pd.DataFrame({'Label': unique_labels, 'Count': counts})
label_counts_df

Number of unique labels: 209297


,Label,Count
0,-1,742579
1,0,8
2,1,47
3,2,6
4,3,31
...,...,...
209292,209291,4
209293,209292,4
209294,209293,3
209295,209294,3


In [ ]:
label_counts_df.describe()

,Label,Count
count,209297.000000,209297.000000
mean,104647.000000,8.493117
std,60418.983983,1623.161064
min,-1.000000,2.000000
25%,52323.000000,2.000000
50%,104647.000000,3.000000
75%,156971.000000,6.000000
max,209295.000000,742579.000000


In [ ]:
label_counts_df[label_counts_df['Count'] == 1].shape

(0, 2)

In [ ]:
np.save(f"{output_dir}/hdbscan_{n_components}_labels.npy", hdbscan_model.labels_)

In [ ]:
from joblib import dump

dump(hdbscan_model, f"{output_dir}/hdbscan_{n_components}_model.joblib")

['/content/drive/MyDrive/logo_recognition_similarity_search_project/output/hdbscan_600_model.joblib']

In [ ]:
from joblib import load

hdbscan_model = load(f"{output_dir}/hdbscan_{n_components}_model.joblib")
hdbscan_model.labels_.shape

(1777584,)



---



### Outliers

In [ ]:
outlier_indices = np.where(hdbscan_model.labels_ == -1)[0]
reduced_embeddings_outliers = reduced_embeddings[outlier_indices]
reduced_embeddings_outliers.shape

(742579, 600)

In [ ]:
from cuml.cluster.hdbscan import HDBSCAN

# A higher min_cluster_size will generate fewer clusters.
# A lower min_cluster_size will generate more clusters.

hdbscan_outliers_model = HDBSCAN(min_cluster_size=2,
                                cluster_selection_epsilon=0.005,
                                metric='euclidean',
                                cluster_selection_method='eom',
                                prediction_data=True,
                                )

In [ ]:
hdbscan_outliers_model.fit(reduced_embeddings_outliers)

HDBSCAN()

In [ ]:
hdbscan_outliers_model.n_clusters_

14

In [ ]:
unique_outlier_labels, outlier_counts = np.unique(hdbscan_outliers_model.labels_, return_counts=True)
print(f"Number of unique labels: {len(unique_outlier_labels)}")
outlier_label_counts_df = pd.DataFrame({'Label': unique_outlier_labels, 'Count': outlier_counts})
outlier_label_counts_df

Number of unique labels: 15


,Label,Count
0,-1,17
1,0,4
2,1,742468
3,2,3
4,3,12
5,4,3
6,5,9
7,6,7
8,7,5
9,8,5


In [ ]:
outlier_label_counts_df.describe()

,Label,Count
count,15.000000,15.000000
mean,6.000000,49505.266667
std,4.472136,191702.366186
min,-1.000000,3.000000
25%,2.500000,4.500000
50%,6.000000,6.000000
75%,9.500000,13.000000
max,13.000000,742468.000000


In [ ]:
soft_clusters = cuml.cluster.hdbscan.membership_vector(hdbscan_outliers_model, reduced_embeddings_outliers[hdbscan_outliers_model.labels_ == -1])
soft_clusters.shape

(17, 14)

In [ ]:
argmax_propability = np.argmax(soft_clusters, axis=1)
argmax_propability

array([ 1, 12,  1,  1, 12,  4,  1,  1,  1,  1,  7, 13, 13,  1,  1,  1,  1])

In [ ]:
outliers_labels = hdbscan_outliers_model.labels_
outliers_labels[outliers_labels ==-1] = argmax_propability
outliers_labels.shape

(742579,)

In [ ]:
unique_outlier_labels, outlier_counts = np.unique(outliers_labels, return_counts=True)
print(f"Number of unique labels: {len(unique_outlier_labels)}")
outlier_label_counts_df = pd.DataFrame({'Label': unique_outlier_labels, 'Count': outlier_counts})
outlier_label_counts_df

Number of unique labels: 14


,Label,Count
0,0,4
1,1,742479
2,2,3
3,3,12
4,4,4
5,5,9
6,6,7
7,7,6
8,8,5
9,9,5


In [ ]:
outlier_label_counts_df.describe()

,Label,Count
count,14.0000,14.000000
mean,6.5000,53041.357143
std,4.1833,198433.803826
min,0.0000,3.000000
25%,3.2500,5.000000
50%,6.5000,6.000000
75%,9.7500,11.250000
max,13.0000,742479.000000


In [ ]:
np.save(f"{output_dir}/hdbscan_{n_components}_outliers_labels.npy", outliers_labels)

In [ ]:
from joblib import dump

dump(hdbscan_outliers_model, f"{output_dir}/hdbscan_{n_components}_outliers_model.joblib")

['/content/drive/MyDrive/logo_recognition_similarity_search_project/output/hdbscan_600_outliers_model.joblib']

In [ ]:
from joblib import load

hdbscan_outliers_model = load(f"{output_dir}/hdbscan_{n_components}_outliers_model.joblib")
hdbscan_outliers_model.labels_.shape

(742579,)



---



In [ ]:
outliers_labels.shape

(742579,)

In [ ]:
hdbscan_outliers_model.n_clusters_,hdbscan_model.n_clusters_

(14, 209296)

In [ ]:
outlier_label_counts_df

,Label,Count
0,0,4
1,1,742479
2,2,3
3,3,12
4,4,4
5,5,9
6,6,7
7,7,6
8,8,5
9,9,5


In [ ]:
outliers_labels.shape

(742579,)

In [ ]:
outliers_labels = outliers_labels+hdbscan_model.n_clusters_+1
outliers_labels.min(),outliers_labels.max()

(209297, 209310)

In [ ]:
fainl_labels = hdbscan_model.labels_
fainl_outlier_indices = np.where(hdbscan_model.labels_ == -1)[0]

fainl_labels[fainl_outlier_indices] = outliers_labels
fainl_labels.shape,fainl_labels.min(),fainl_labels.max()

((1777584,), 0, 209310)

In [ ]:
unique_labels, counts = np.unique(fainl_labels, return_counts=True)
print(f"Number of unique labels: {len(unique_labels)}")
label_counts_df = pd.DataFrame({'Label': unique_labels, 'Count': counts})
label_counts_df

Number of unique labels: 209310


,Label,Count
0,0,8
1,1,47
2,2,6
3,3,31
4,4,8
...,...,...
209305,209306,5
209306,209307,6
209307,209308,17
209308,209309,6


In [ ]:
label_counts_df.describe()

,Label,Count
count,209310.000000,209310.000000
mean,104654.500067,8.492590
std,60422.736875,1622.892081
min,0.000000,2.000000
25%,52327.250000,2.000000
50%,104654.500000,3.000000
75%,156981.750000,6.000000
max,209310.000000,742479.000000


In [ ]:
np.save(f"{output_dir}/hdbscan_{n_components}_fainl_labels.npy", fainl_labels)

## Save

This section saves the clustering results to google drive.  These files will be used in subsequent notebooks for analysis and visualization.  The soft clusters represent the probability of each data point belonging to each cluster, providing a measure of uncertainty in the cluster assignments.

In [ ]:
ds = ds.add_column('category', fainl_labels.tolist())

In [ ]:
ds

Dataset({
    features: ['id', 'path', 'prompt', 'prompt_embedding', 'image_embedding', 'category'],
    num_rows: 1777584
})

In [ ]:
ds.features

{'id': Value(dtype='string', id=None),
 'path': Value(dtype='string', id=None),
 'prompt': Value(dtype='string', id=None),
 'prompt_embedding': Sequence(feature=Value(dtype='float32', id=None), length=-1, id=None),
 'image_embedding': Sequence(feature=Value(dtype='float32', id=None), length=-1, id=None),
 'category': Value(dtype='int64', id=None)}

In [ ]:
df = ds.remove_columns(['image_embedding', 'prompt_embedding']).to_pandas()
df.head()

,id,path,prompt,category
0,1104391803911815278,images/00/00/1104391803911815278_0_0.png,cute simple baby shark logo for company,197906
1,1104391803911815278,images/00/00/1104391803911815278_0_1.png,cute simple baby shark logo for company,197906
2,1104391803911815278,images/00/00/1104391803911815278_1_0.png,cute simple baby shark logo for company,197906
3,1104391803911815278,images/00/00/1104391803911815278_1_1.png,cute simple baby shark logo for company,197906
4,1132464139621646377,images/00/00/1132464139621646377_0_0.png,"food truck, sells french fries, logo on side i...",127006


In [ ]:
df.shape

(1777584, 4)

In [ ]:
df.to_csv(f"{output_dir}/logo-recognition-with-{n_components}-hdbscan-category.tsv", sep='\t', index=False)

In [ ]:
df = pd.read_csv(f"{output_dir}/logo-recognition-with-{n_components}-hdbscan-category.tsv", sep='\t')
df.head()

,id,path,prompt,category
0,1104391803911815278,images/00/00/1104391803911815278_0_0.png,cute simple baby shark logo for company,197906
1,1104391803911815278,images/00/00/1104391803911815278_0_1.png,cute simple baby shark logo for company,197906
2,1104391803911815278,images/00/00/1104391803911815278_1_0.png,cute simple baby shark logo for company,197906
3,1104391803911815278,images/00/00/1104391803911815278_1_1.png,cute simple baby shark logo for company,197906
4,1132464139621646377,images/00/00/1132464139621646377_0_0.png,"food truck, sells french fries, logo on side i...",127006




---

